# User Behavior Prediction in Food Delivery Applications
## COE546 - Machine Learning Project
### *Lara Snih · Jana Antoun · Ghadi Ammar*

---
## Section 1 - Data Loading & Exploratory Data Analysis

In [ ]:
# ============================================================
# CELL 0 - Imports & Configuration
# ============================================================
# All libraries imported upfront for clarity and reproducibility.

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ============================================================
# CELL 1 - Load Raw Data
# ============================================================
# Load train and test sets. Preserve test IDs for the final submission file.

train_df = pd.read_csv('/kaggle/input/competitions/user-behavior-prediction-in-food-delivery-applications/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/user-behavior-prediction-in-food-delivery-applications/test.csv')
test_ids = test_df['id'].copy()

print(f'Train : {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'Test  : {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
print(f'\nTarget distribution:')
print(train_df['order_placed'].value_counts())
print(f'\nPositive rate : {train_df["order_placed"].mean():.4%}')


Train : 297,236 rows × 18 columns
Test  : 99,639 rows × 17 columns

Target distribution:
order_placed
0    288592
1      8644
Name: count, dtype: int64

Positive rate : 2.9081%


In [ ]:
# ============================================================
# CELL 2 - Missing Values
# ============================================================
# Identify column types and missing value patterns before any preprocessing.
# Key observation: f12, f13, f14, f15, and f17 are missing together as a block,
# suggesting they are all offer-related fields absent when no promotion was shown.

print('=== TRAIN SCHEMA ===')
train_df.info()

print('\n=== MISSING VALUES ===')
missing     = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False))


=== TRAIN SCHEMA ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 297236 entries, 0 to 297235
Data columns (total 18 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            297236 non-null  int64  
 1   f2            297236 non-null  object 
 2   f3            297236 non-null  object 
 3   f4            297236 non-null  object 
 4   f5            297236 non-null  object 
 5   f6            297236 non-null  int64  
 6   f7            297236 non-null  object 
 7   f8            297236 non-null  int64  
 8   f9            297236 non-null  object 
 9   f10           297236 non-null  int64  
 10  f11           297236 non-null  float64
 11  f12           206718 non-null  object 
 12  f13           206718 non-null  float64
 13  f14           206718 non-null  float64
 14  f15           206718 non-null  float64
 15  f16           297236 non-null  float64
 16  f17           206718 non-null  object 
 17  order_placed  297236 non-nu

In [ ]:
# ============================================================
# CELL 3 - Target Distribution
# ============================================================
# The target is severely imbalanced: only 2.9% of sessions result in an order.
# A model predicting class 0 always would score 97% accuracy but AUC of 0.5.
# This confirms AUC is the correct evaluation metric for this task.

print(f'Positive rate : {train_df["order_placed"].mean():.4%}')
print(f'Negative rate : {1 - train_df["order_placed"].mean():.4%}')
print(f'Imbalance ratio (neg:pos) : {(train_df["order_placed"]==0).sum() / (train_df["order_placed"]==1).sum():.1f}:1')
print(train_df['order_placed'].value_counts())


Positive rate : 2.9081%
Negative rate : 97.0919%
Imbalance ratio (neg:pos) : 33.4:1
order_placed
0    288592
1      8644
Name: count, dtype: int64


--
## Section 2 - Feature Engineering

Each engineered feature was designed to represent a meaningful behavioral signal, guided by the question: why did a user place an order, or decide not to?

* Temporal features - session_duration_s, time_to_event_s, and remaining_session_s capture how long a user stayed in a session and how quickly they took action.
* Cyclical time encoding - hour_sin/cos and dow_sin/cos preserve the natural continuity of time. For example, 11 PM and midnight are treated as close together, not far apart. Raw integers alone would not capture this.
* Cart features - log_cart_value, avg_item_value, has_items, and cart_value_rate reflect purchase intent based on the current cart state.
* Offer features - discount_ratio, eligible_for_offer, threshold_gap, and offer_generosity measure whether a promotion is likely to be appealing to the user.
* Session behavior - event_position and is_fast_action capture how the user moved through the session.
* User-type features -is_returning, returning_with_cart, and returning_with_offer account for behavioral differences between returning and new customers.
* Promo history- promo_acceptance_rate and log_f15 estimate how responsive a user has been to past promotions.
* Missingness indicator - a missing_block flag explicitly captures when all offer fields are absent together, treating it as a meaningful signal rather than noise.

In [ ]:
# ============================================================
# CELL 4 - Feature Engineering
# ============================================================
# All transformations are applied identically to train and test to prevent leakage.

def engineer_features(df):
    df = df.copy()

    # ── 1. Datetimes ─────────────────────────────────────────
    for col in ['f3', 'f4', 'f5']:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

    df['session_duration_s']  = (df['f4'] - df['f3']).dt.total_seconds().clip(1, 10_000)
    df['time_to_event_s']     = (df['f5'] - df['f3']).dt.total_seconds().clip(0, 10_000)
    df['remaining_session_s'] = (df['session_duration_s'] - df['time_to_event_s']).clip(0)

    # ── 2. Cyclical time encoding ─────────────────────────────
    # hour 23 and hour 0 are 1 apart in reality — sin/cos encoding preserves this.
    hour = df['f3'].dt.hour
    dow  = df['f3'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    df['dow_sin']  = np.sin(2 * np.pi * dow  / 7)
    df['dow_cos']  = np.cos(2 * np.pi * dow  / 7)

    # ── 3. Cart / value signals ───────────────────────────────
    # f10 = items in cart, f11 = cart value
    # Log transforms applied because both are right-skewed.
    df['log_cart_value']  = np.log1p(df['f11'].clip(0))
    df['log_cart_items']  = np.log1p(df['f10'].clip(0))
    df['avg_item_value']  = (df['f11'] / (df['f10'] + 1)).clip(0, 1000)
    df['has_items']       = (df['f10'] > 0).astype(int)
    df['cart_value_rate'] = df['f11'] / (df['time_to_event_s'] + 1)

    # ── 4. Offer signals ──────────────────────────────────────
    # f12 = offer type, f13 = discount, f14 = min threshold, f15 = promo count
    df['has_offer']          = df['f12'].notna().astype(int)
    df['log_discount']       = np.log1p(df['f13'].fillna(0).clip(0))
    df['discount_ratio']     = (df['f13'].fillna(0) / (df['f11'] + 1)).clip(0, 10)
    df['eligible_for_offer'] = np.where(df['f14'].notna(), (df['f11'] >= df['f14']).astype(int), 0)
    df['threshold_gap']      = (df['f11'] - df['f14'].fillna(df['f11'])).clip(-500, 500)
    df['pct_to_threshold']   = (df['f11'] / (df['f14'].fillna(df['f11']) + 1)).clip(0, 5)
    df['offer_generosity']   = (df['f13'].fillna(0) / (df['f14'].fillna(1) + 1)).clip(0, 5)

    # ── 5. Session behavior ───────────────────────────────────
    df['event_position'] = (df['time_to_event_s'] / (df['session_duration_s'] + 1)).clip(0, 1)
    df['is_fast_action'] = (df['time_to_event_s'] < 300).astype(int)
    df['log_f6']         = np.log1p(df['f6'].clip(0))
    df['log_f16']        = np.log1p(df['f16'].clip(0))

    # ── 6. User type signals ──────────────────────────────────
    # f9: OC = returning customer, NC = new customer
    df['is_returning']         = (df['f9'] == 'OC').astype(int)
    df['returning_with_cart']  = df['is_returning'] * df['has_items']
    df['returning_with_offer'] = df['is_returning'] * df['has_offer']

    # ── 7. Promo history ──────────────────────────────────────
    # f8 = declined offers, f15 = promos shown
    df['promo_acceptance_rate'] = df['f8'] / (df['f15'].fillna(0) + 1)
    df['log_f15']               = np.log1p(df['f15'].fillna(0).clip(0))

    # ── 8. Missingness indicator ──────────────────────────────
    # All five offer columns missing together = no promotion was shown.
    # Encoding this explicitly lets the model learn from it directly.
    offer_cols = ['f12', 'f13', 'f14', 'f15', 'f17']
    df['missing_block'] = df[offer_cols].isnull().all(axis=1).astype(int)

    return df


train_df = engineer_features(train_df)
test_df  = engineer_features(test_df)

print(f'Train shape after engineering : {train_df.shape}')
print(f'Test shape after engineering  : {test_df.shape}')


Train shape after engineering : (297236, 47)
Test shape after engineering  : (99639, 46)


In [ ]:
# ============================================================
# CELL 5 - Prepare Model Inputs
# ============================================================
# Dropped columns:
#   id, f2       — identifiers, no predictive value
#   f3, f4, f5   — replaced by engineered time features
#   f17          — offer outcome, excluded to prevent leakage
#   order_placed — target variable
#
# Categorical columns (f7, f9, f12) are passed directly to
# CatBoost which handles them natively. LightGBM receives them as .astype('category').

drop_cols    = ['id', 'f2', 'f3', 'f4', 'f5', 'f17', 'order_placed']
cat_cols     = ['f7', 'f9', 'f12']
feature_cols = [c for c in train_df.columns if c not in drop_cols and c != 'order_placed']

X      = train_df[feature_cols].copy()
y      = train_df['order_placed'].copy()
X_test = test_df[feature_cols].copy()

# Fill categorical NaNs with sentinel 'NONE'
# (represents rows where no offer was shown)
for col in cat_cols:
    X[col]      = X[col].fillna('NONE')
    X_test[col] = X_test[col].fillna('NONE')

print(f'Total features  : {len(feature_cols)}')
print(f'Categorical     : {cat_cols}')
print(f'X shape         : {X.shape}')
print(f'Positive rate   : {y.mean():.4%}')
print(f'NaNs in X       : {X.isnull().sum().sum()}')
print(f'NaNs in X_test  : {X_test.isnull().sum().sum()}')


Total features  : 40
Categorical     : ['f7', 'f9', 'f12']
X shape         : (297236, 40)
Positive rate   : 2.9081%
NaNs in X       : 271554
NaNs in X_test  : 90012


---
## Section 3 - Model Training

### Why These Models?

CatBoost  works especially well with categorical features because it encodes them internally using ordered target statistics, avoiding leakage while eliminating the need for manual encoding. Its built-in regularization and balanced tree growth make it strong and reliable out of the box.

LightGBM is designed for speed and efficiency on large datasets, using leaf-wise tree growth and histogram-based splitting. It handles categoricals natively via the .astype('category') conversion, and its different growth strategy from CatBoost makes it a valuable ensemble partner.

### Why 5-Fold Stratified Cross-Validation?

A single train-validation split gives only one estimate of performance, which can be sensitive to how the data was divided. Using 5-fold cross-validation ensures every sample is used for validation exactly once, giving a more reliable and unbiased estimate of generalization.

Stratification is critical here: with only 2.9% positive samples, unstratified splits could produce folds with very different class ratios. Stratified folds preserve the original 97:3 distribution in every split.

Test predictions are averaged across all five folds, which also reduces variance in the final output.

In [ ]:
# ============================================================
# CELL 6 - Cross-Validation Setup
# ============================================================
# Stratified to preserve the 2.9% positive class ratio in every fold.

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('StratifiedKFold: 5 folds, shuffled, random_state=42')
print('Each fold preserves the 2.9% positive class ratio.')


StratifiedKFold: 5 folds, shuffled, random_state=42
Each fold preserves the 2.9% positive class ratio.


In [ ]:
# ============================================================
# CELL 7 - CatBoost (5-Fold CV)
# ============================================================
# Key hyperparameters:
#   depth=7              — balances expressiveness vs overfitting
#   l2_leaf_reg=5        — L2 regularization on leaf weights
#   subsample=0.8        — 80% row sampling per tree
#   colsample_bylevel=0.8 — 80% feature sampling per level
#   min_data_in_leaf=20  — prevents memorising single examples
#   early_stopping=150   — stops when val AUC plateaus

from catboost import CatBoostClassifier, Pool
import numpy as np

oof_cb        = np.zeros(len(X))
test_preds_cb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f'\n── Fold {fold+1} ──────────────────────')

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    train_pool = Pool(X_tr,   y_tr,  cat_features=cat_cols)
    val_pool   = Pool(X_val,  y_val, cat_features=cat_cols)
    test_pool  = Pool(X_test,        cat_features=cat_cols)

    model_cb = CatBoostClassifier(
        iterations            = 3000,
        learning_rate         = 0.05,
        depth                 = 7,
        loss_function         = 'Logloss',
        eval_metric           = 'AUC',
        random_seed           = 42,
        l2_leaf_reg           = 5,
        subsample             = 0.8,
        colsample_bylevel     = 0.8,
        min_data_in_leaf      = 20,
        verbose               = 200,
        early_stopping_rounds = 150,
    )
    model_cb.fit(train_pool, eval_set=val_pool)

    oof_cb[val_idx]  = model_cb.predict_proba(X_val)[:, 1]
    test_preds_cb   += model_cb.predict_proba(test_pool)[:, 1] / kf.n_splits

    print(f'Fold AUC: {roc_auc_score(y_val, oof_cb[val_idx]):.6f}')

print(f'\n{"="*40}')
print(f'CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.6f}')



── Fold 1 ──────────────────────
0:	test: 0.9205490	best: 0.9205490 (0)	total: 200ms	remaining: 10m


In [ ]:
# ============================================================
# CELL 8 - LightGBM (5-Fold CV)
# ============================================================
# Categorical columns converted to .astype('category') so
# LightGBM handles them natively.
# Key hyperparameters match CatBoost for a fair comparison:
#   num_leaves=63        — leaf-wise growth, ~depth 6
#   min_child_samples=20 — minimum rows per leaf
#   subsample=0.8        — row and feature subsampling
#   reg_lambda=5.0       — L2 regularization

import lightgbm as lgb

X_lgb      = X.copy()
X_test_lgb = X_test.copy()
for col in cat_cols:
    X_lgb[col]      = X_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')

oof_lgb        = np.zeros(len(X_lgb))
test_preds_lgb = np.zeros(len(X_test_lgb))

params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        63,
    'max_depth':         8,
    'min_child_samples': 20,
    'subsample':         0.8,
    'colsample_bytree':  0.8,
    'reg_lambda':        5.0,
    'random_state':      42,
    'verbose':           -1,
}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_lgb, y)):
    print(f'\n── Fold {fold+1} ──────────────────────')

    X_tr,  X_val  = X_lgb.iloc[tr_idx], X_lgb.iloc[val_idx]
    y_tr,  y_val  = y.iloc[tr_idx],     y.iloc[val_idx]

    train_ds = lgb.Dataset(X_tr,  label=y_tr)
    val_ds   = lgb.Dataset(X_val, label=y_val, reference=train_ds)

    model_lgb = lgb.train(
        params, train_ds,
        num_boost_round = 3000,
        valid_sets      = [val_ds],
        callbacks       = [lgb.early_stopping(150, verbose=False),
                           lgb.log_evaluation(200)],
    )

    oof_lgb[val_idx]  = model_lgb.predict(X_val)
    test_preds_lgb   += model_lgb.predict(X_test_lgb) / kf.n_splits

    print(f'Fold AUC: {roc_auc_score(y_val, oof_lgb[val_idx]):.6f}')

print(f'\n{"="*40}')
print(f'LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.6f}')


---
## Section 4 - Ensemble & Final Submission

The final prediction combines CatBoost and LightGBM into a weighted ensemble. These two models use different tree-building strategies and categorical handling approaches, which means they tend to make different kinds of errors. Averaging their predictions allows some of those errors to cancel out, producing a more stable and accurate result than either model alone.

The optimal blend weight is found by grid searching over all CatBoost/LightGBM weight combinations using only out-of-fold predictions. The test set is never used during this search, so there is no data leakage.

In [ ]:
# ============================================================
# CELL 9 - Ensemble: Find Optimal Blend Weight
# ============================================================
# Grid search from 0.0 to 1.0 in steps of 0.05.
# Selects the weight that maximises blended OOF AUC.
# Test set is never touched during this search.

best_auc    = 0
best_weight = 0

for w in np.arange(0.0, 1.01, 0.05):
    blended = w * oof_cb + (1 - w) * oof_lgb
    auc     = roc_auc_score(y, blended)
    if auc > best_auc:
        best_auc    = auc
        best_weight = w

print(f'CatBoost OOF AUC  : {roc_auc_score(y, oof_cb):.6f}')
print(f'LightGBM OOF AUC  : {roc_auc_score(y, oof_lgb):.6f}')
print(f'\nBest blend weight : {best_weight:.2f} CatBoost / {1-best_weight:.2f} LightGBM')
print(f'Best ensemble AUC : {best_auc:.6f}')


In [ ]:
# ============================================================
# CELL 10 - Save Ensemble Submission
# ============================================================
# Apply the same blend weights found above to the test predictions.

test_preds_ensemble = best_weight * test_preds_cb + (1 - best_weight) * test_preds_lgb

submission = pd.DataFrame({
    'id':           test_ids,
    'order_placed': test_preds_ensemble,
})
submission.to_csv('submission.csv', index=False)

print(f'Shape  : {submission.shape}')
print(f'\nSample predictions:')
print(submission.head(10).to_string(index=False))
print(f'\nPrediction range : {test_preds_ensemble.min():.6f} → {test_preds_ensemble.max():.6f}')
print('\n submission.csv saved')
